In [ ]:
# !pip install safetensors wtpsplit -q
!pip install fairseq2 --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cu124 -q
!pip install sonar-space -q

import gc
import torch
import random
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline, EmbeddingToTextModelPipeline
import torch.nn.functional as F

t2t_model_emb = TextToEmbeddingModelPipeline(
    encoder="text_sonar_basic_encoder",
    tokenizer="text_sonar_basic_encoder",
    device=torch.device("cuda"),
    dtype=torch.float16,
)

t2t_model_dec = EmbeddingToTextModelPipeline(
    decoder="text_sonar_basic_decoder",
    tokenizer="text_sonar_basic_encoder",
    device=torch.device("cuda"),
    dtype=torch.float16,
)

del t2t_model_emb, t2t_model_dec
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import torch
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM 
import torch
from torch.utils.data import IterableDataset, DataLoader
import random
import torch
import torch.nn.functional as F
from tqdm import tqdm
import os
import random

In [ ]:
from datasets import load_dataset

device = "cuda"
dataset = load_dataset("Sibgat-Ul/c4_fineWebMath", streaming=True)
dataset = dataset["train"]

In [ ]:
first_embedding = None
for ex in dataset:
    if ex["sonar_embeddings"] and len(ex["sonar_embeddings"]) > 0:
        first_embedding = ex["sonar_embeddings"][0]
        d_sonar = len(first_embedding)
        break
if first_embedding is None:
    raise ValueError("No embeddings generated in the dataset. Check your data and preprocessing.")
print(f"Determined d_sonar from processed dataset: {d_sonar}")

print("Collecting sample embeddings from processed dataset for robust scaler calculation...")
all_sampled_embeddings_for_scaler = []

sampled_dataset_for_scaler = dataset.take(500)

for ex in tqdm(sampled_dataset_for_scaler, desc="Sampling embeddings for scaler"):
    if ex["sonar_embeddings"]:
        all_sampled_embeddings_for_scaler.append(
            torch.tensor(ex["sonar_embeddings"], dtype=torch.float32).cpu()
        )

if not all_sampled_embeddings_for_scaler:
    raise ValueError("No embeddings collected for scaler calculation. Check your dataset and sampling.")

combined_embeddings_for_scaler = torch.cat(all_sampled_embeddings_for_scaler, dim=0)

scaler_median = torch.median(combined_embeddings_for_scaler, dim=0).values
print(scaler_median)
q3 = torch.quantile(combined_embeddings_for_scaler, 0.75, dim=0)
q1 = torch.quantile(combined_embeddings_for_scaler, 0.25, dim=0)
scaler_iqr = q3 - q1

scaler_median = scaler_median.to(device)
scaler_iqr = scaler_iqr.to(device)

print(f"Calculated d_sonar: {d_sonar}")
print(f"Calculated scaler_median shape: {scaler_median.shape}")
print(f"Calculated scaler_iqr shape: {scaler_iqr.shape}")
print(f"Total embeddings collected for scaler: {combined_embeddings_for_scaler.shape[0]}")

del combined_embeddings_for_scaler
del all_sampled_embeddings_for_scaler

In [ ]:
class PreNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, scaler_mean, scaler_std):
        super().__init__()
        self.linear = nn.Linear(input_dim, hidden_dim, bias=True)
        # Convert scaler parameters to float16
        self.register_buffer('scaler_mean', scaler_mean.to(torch.float16))
        self.register_buffer('scaler_std', scaler_std.to(torch.float16))
        # Convert linear layer to float16
        self.linear = self.linear.half()

    def normalize(self, x):
        # Ensure input is float16
        x = x.to(torch.float16)
        return (x - self.scaler_mean) / (self.scaler_std + 1e-6)

    def forward(self, x):
        x = self.normalize(x)
        return self.linear(x)

class PostNet(nn.Module):
    def __init__(self, hidden_dim, output_dim, scaler_mean, scaler_std):
        super().__init__()
        self.linear = nn.Linear(hidden_dim, output_dim, bias=True)
        # Convert scaler parameters to float16
        self.register_buffer('scaler_mean', scaler_mean.to(torch.float16))
        self.register_buffer('scaler_std', scaler_std.to(torch.float16))
        # Convert linear layer to float16
        self.linear = self.linear.half()

    def denormalize(self, x):
        # Ensure input is float16
        x = x.to(torch.float16)
        return x * (self.scaler_std + 1e-6) + self.scaler_mean

    def forward(self, x):
        return self.denormalize(self.linear(x))

class QwenLCM(nn.Module):
    def __init__(self, d_sonar, scaler_mean, scaler_std, qwen_model_name="Qwen/Qwen3-0.6B"):
        super().__init__()

        self.qwen = AutoModelForCausalLM.from_pretrained(qwen_model_name)

        for param in self.qwen.parameters():
            param.requires_grad = False
    
        for name, param in self.qwen.named_parameters():
            if "model.norm" in name:
                param.requires_grad = True

        self.qwen.model.embed_tokens = nn.Identity()
        self.qwen.lm_head = nn.Identity()

        qwen_hidden_size = 1024
        self.prenet = PreNet(input_dim=d_sonar, hidden_dim=qwen_hidden_size, scaler_mean=scaler_mean, scaler_std=scaler_std)
        self.postnet = PostNet(hidden_dim=qwen_hidden_size, output_dim=d_sonar, scaler_mean=scaler_mean, scaler_std=scaler_std)
        
        # Convert all parameters to float16
        self.half()

    def train_layers(self):
        for name, param in self.qwen.named_parameters():
            if any(f"layers.{i}." in name for i in [0, 2, 4, 6, 9, 12, 15, 19, 24, 28]) or name == "norm.weight":
                param.requires_grad = True

    def freeze_pre_post_net(self):
        for param in self.prenet.parameters():
            param.requires_grad = False
        for param in self.postnet.parameters():
            param.requires_grad = False

    def forward(self, sonar_seq, attention_mask=None):
        """
        sonar_seq: (batch_size, seq_len, d_sonar) - sequence of SONAR embeddings
        """
        # Ensure input is float16
        sonar_seq = sonar_seq.to(torch.float16)
        hidden = self.prenet(sonar_seq)  
        qwen_output = self.qwen.model(inputs_embeds=hidden, attention_mask=attention_mask).last_hidden_state
        
        # Project back to SONAR embedding space
        predicted_sonar_seq = self.postnet(qwen_output)
        
        return predicted_sonar_seq

In [ ]:
class SentenceSequenceDataset(IterableDataset):
    def __init__(self, hf_dataset, max_len=64):
        self.hf_dataset = hf_dataset
        self.max_len = max_len

    def __iter__(self):
        for ex in self.hf_dataset:
            embs = ex["sonar_embeddings"]
            if len(embs) < 2:
                continue

            max_idx = min(len(embs) - 1, self.max_len - 1)
            input_seq = torch.tensor(embs[:max_idx], dtype=torch.float16)
            target_seq = torch.tensor(embs[1:max_idx + 1], dtype=torch.float16)
            yield (input_seq, target_seq)

def collate_fn(batch):
    x_seqs, y_seqs = zip(*batch)
    max_len = max(seq.shape[0] for seq in x_seqs)
    dim = x_seqs[0].shape[-1]

    padded_inputs = torch.zeros(len(batch), max_len, dim)
    padded_targets = torch.zeros(len(batch), max_len, dim)
    attention_mask = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for i, (x, y) in enumerate(zip(x_seqs, y_seqs)):
        padded_inputs[i, :x.shape[0]] = x
        padded_targets[i, :y.shape[0]] = y
        attention_mask[i, :x.shape[0]] = 1

    return padded_inputs, attention_mask, padded_targets

train_dataset = SentenceSequenceDataset(dataset, max_len=64)
# train_dataset = ChunkedStreamingDataset(dataset, max_len=64)

# Create DataLoader
train_loader = DataLoader(
    train_dataset, 
    batch_size=16, 
    collate_fn=collate_fn,
    # Note: shuffle=True doesn't work with IterableDataset
    # Shuffling is handled within the dataset classes above
)

# Training loop
lcm = QwenLCM(d_sonar=1024, scaler_mean=scaler_median, scaler_std=scaler_iqr).to(device)
optimizer = torch.optim.AdamW(lcm.parameters(), lr=1e-7, eps=1e-06, weight_decay=0.1)

In [ ]:
from transformers import GenerationConfig

lcm.qwen.generation_config = GenerationConfig(max_length=11)
lcm.qwen.generation_config 

In [ ]:

# model.load_state_dict(torch.load("/kaggle/input/base-lcm/pytorch/default/4/checkpoint_epoch_3.pt", weights_only=True)["model_state_dict"])
lcm.train_layers()
lcm.freeze_pre_post_net()

In [ ]:
sum([p.numel() for name, p in lcm.named_parameters() if p.requires_grad])

In [ ]:
eval_doc = """Climate change poses one of the most significant challenges facing humanity in the 21st century. Rising global temperatures are causing ice caps to melt, leading to sea level rise and coastal flooding. Extreme weather events are becoming more frequent and severe, affecting agriculture and human settlements."""

eval_sentences = [
    "Tomorrow I have exam at 8 am.",
    "I just had my dinner and will revise the lessons before I sleep.",
    "To reach early, I will have to catch the bus on the morning.",
    "As I have to catch the bus early, I will have to sleep as soon as I finish the lessons."
]


In [ ]:
import os
import torch
import torch.nn.functional as F
from tqdm import tqdm
import gc
from itertools import cycle

max_iterations = 10000  # Total number of training steps/iterations
eval_every_n_iters = 500  # Set to >0 if you plan to evaluate every N iterations
save_every_n_iters = 5000  # Save model every N iterations

train_iterator = iter(cycle(train_loader))  # Infinite iterator over train_loader

best_similarity = -1.0
training_history = []

total_loss = 0.0
total_similarity = 0.0
num_batches = 0

# Initialize the model and ensure it's in float16
lcm = lcm.to(device).half()
optimizer = torch.optim.AdamW(lcm.parameters(), lr=1e-7, eps=1e-06, weight_decay=0.1)

pbar = tqdm(range(1, max_iterations + 1), desc="Training")

for iteration in pbar:
    lcm.train()

    # Load batch and ensure it's in float16
    inputs, mask, targets = next(train_iterator)
    inputs = inputs.to(device).half()  # Convert to float16
    mask = mask.to(device)
    targets = targets.to(device).half()  # Convert to float16

    # Forward pass
    predictions = lcm(inputs, attention_mask=mask)
    predicted_embeddings = predictions['embeddings']
    
    # Compute MSE loss
    mse_loss = F.mse_loss(predicted_embeddings, targets)
    cosine_sim = F.cosine_similarity(predicted_embeddings.float(), targets.float(), dim=-1).mean()

    # Backward pass
    mse_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    total_loss += mse_loss.item()
    total_similarity += cosine_sim.item()
    num_batches += 1

    pbar.set_postfix({"mse_loss": f"{mse_loss.item():.5f}", "cos_sim": f"{cosine_sim.item():.2f}"})

    if iteration % eval_every_n_iters == 0:
        print("Evaluating: ")
        def evaluate_model():
            import torch
            import random
            from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline, EmbeddingToTextModelPipeline
            import torch.nn.functional as F

            try:
                eval_doc = """Climate change poses one of the most significant challenges facing humanity in the 21st century. Rising global temperatures are causing ice caps to melt, leading to sea level rise and coastal flooding. Extreme weather events are becoming more frequent and severe, affecting agriculture and human settlements."""
        
                eval_sentences = [
                    "Tomorrow I have exam at 8 am.",
                    "I just had my dinner and will revise the lessons before I sleep.",
                    "To reach early, I will have to catch the bus on the morning.",
                    "As I have to catch the bus early, I will have to sleep as soon as I finish the lessons."
                ]
        
                lcm.eval()
            
                t2t_model_emb = TextToEmbeddingModelPipeline(
                    encoder="text_sonar_basic_encoder",
                    tokenizer="text_sonar_basic_encoder",
                    device=torch.device("cuda"),
                    dtype=torch.float16,  # Explicitly set dtype to float16
                )
                
                t2t_model_dec = EmbeddingToTextModelPipeline(
                    decoder="text_sonar_basic_decoder",
                    tokenizer="text_sonar_basic_encoder",
                    device=torch.device("cuda"),
                    dtype=torch.float16,  # Explicitly set dtype to float16
                )
            
                # No need to call half() here since model is already in float16
                print("-" * 50)
                
                total_eval_loss = 0.0
                total_eval_similarity = 0.0
            
                with torch.no_grad():
                    selected_sentence = eval_sentences[-1]
                    print(f"Evaluating on: '{selected_sentence[:50]}...'")
                
                    target_embedding = t2t_model_emb.predict([selected_sentence], source_lang="eng_Latn")
                    target_tensor = target_embedding.unsqueeze(0)
                    
                    # Ensure inputs are in float16
                    input_embeddings = t2t_model_emb.predict(eval_sentences, source_lang="eng_Latn").to(torch.float16)
                    pred_embedding = lcm(input_embeddings.unsqueeze(0))['embeddings']
                    
                    # Convert to float32 for loss computation to avoid numerical instability
                    eval_loss = F.mse_loss(pred_embedding.float(), target_tensor.float())
                    eval_similarity = F.cosine_similarity(pred_embedding.float(), target_tensor.float()).mean()
                    
                    total_eval_loss += eval_loss.item()
                    total_eval_similarity += eval_similarity.item()
                
                    try:
                        print(pred_embedding.dtype)
                        decoded_text = t2t_model_dec.predict(pred_embedding.squeeze(0), target_lang="eng_Latn")
                        print(f"Original: {selected_sentence}")
                        print(f"Predicted: {decoded_text}")
                    except Exception as e:
                        print(f"Decoding failed: {e}")
                
                    print(f"Eval Loss: {eval_loss.item():.5f}")
                    print(f"Eval Cosine Similarity: {eval_similarity.item():.4f}")
                    
                print("Eval loss:", total_eval_loss, "Eval cos sim:", total_eval_similarity)
                del t2t_model_emb, t2t_model_dec
                torch.cuda.empty_cache()
                gc.collect()
                lcm.train()  # Set back to training mode
            except Exception as e:
                print(e)
        evaluate_model()
        
    if iteration % save_every_n_iters == 0:
        avg_loss = total_loss / num_batches
        avg_sim = total_similarity / num_batches

        checkpoint = {
            'iteration': iteration,
            'model_state_dict': lcm.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_loss,
            'train_similarity': avg_sim,
            'training_history': training_history
        }

        save_path = os.path.join(save_dir, f"checkpoint_iter_{iteration}.pt")
        torch.save(checkpoint, save_path)
        print(f"\n✅ Model checkpoint saved at: {save_path}")

        total_loss = 0.0
        total_similarity = 0.0
        num_batches = 0

        torch.cuda.empty_cache()
        gc.collect()

avg_loss = total_loss / num_batches
avg_sim = total_similarity / num_batches
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Total iterations: {max_iterations}")
print(f"Final training loss: {avg_loss:.5f}")
print(f"Final training similarity: {avg_sim:.4f}")

torch.cuda.empty_cache()
gc.collect()

In [ ]:
!pip install fairseq2 --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cu124 -q
!pip install sonar-space -q

In [ ]:
def do_eval():
    weights = torch.load("/kaggle/input/base-lcm/pytorch/default/4/checkpoint_epoch_3.pt", weights_only=True)["model_state_dict"]
    # "prenet.scaler_mean", "prenet.scaler_std", "postnet.scaler_mean", "postnet.scaler_std". 
    prenet_mean = weights["prenet.scaler_mean"]
    prenet_std =  weights["prenet.scaler_std"]
    
    postnet_mean = weights["postnet.scaler_mean"]
    postnet_std = weights["postnet.scaler_std"]
    
    lcm = QwenLCM(d_sonar=1024, scaler_mean=prenet_mean, scaler_std=prenet_std).to("cuda")
    lcm.load_state_dict(weights)

    return lcm
    
# model = do_eval()

In [ ]:
def evaluate_model(lcm):
    import torch
    import random
    from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline, EmbeddingToTextModelPipeline
    import torch.nn.functional as F
    """Evaluate model on the given document sentences"""
    eval_doc = """Climate change poses one of the most significant challenges facing humanity in the 21st century. Rising global temperatures are causing ice caps to melt, leading to sea level rise and coastal flooding. Extreme weather events are becoming more frequent and severe, affecting agriculture and human settlements."""

    eval_sentences = [
        "Tomorrow I have exam at 8 am.",
        "I just had my dinner and will revise the lessons before I sleep.",
        "To reach early, I will have to catch the bus on the morning.",
        "As I have to catch the bus early, I will have to sleep as soon as I finish the lessons."
    ]

    
    lcm.eval()
    
    t2t_model_emb = TextToEmbeddingModelPipeline(
        encoder="text_sonar_basic_encoder",
        tokenizer="text_sonar_basic_encoder",
        device=torch.device("cuda"),
        dtype=torch.float16,
    )
    
    t2t_model_dec = EmbeddingToTextModelPipeline(
        decoder="text_sonar_basic_decoder",
        tokenizer="text_sonar_basic_encoder",
        device=torch.device("cuda"),
        dtype=torch.float16,
    )
    
    lcm.half()
    # print(f"\n🔍 Evaluation at Epoch {epoch + 1}")
    print("-" * 50)
    
    total_eval_loss = 0.0
    total_eval_similarity = 0.0
    
    with torch.no_grad():
        selected_sentence = eval_sentences[-1]
        print(f"Evaluating on: '{selected_sentence[:50]}...'")
    
        target_embedding = t2t_model_emb.predict([selected_sentence], source_lang="eng_Latn")
        target_tensor = target_embedding.unsqueeze(0)
        
        pred_embedding = lcm(t2t_model_emb.predict(eval_sentences, source_lang="eng_Latn").unsqueeze(0))
        
        eval_loss = F.mse_loss(pred_embedding[:, -2].unsqueeze(0).float(), target_tensor.float())
        eval_similarity = F.cosine_similarity(pred_embedding, target_tensor).mean()
        
        total_eval_loss += eval_loss.item()
        total_eval_similarity += eval_similarity.item()
        
        try:
            print(pred_embedding.dtype)
            decoded_text = t2t_model_dec.predict(pred_embedding.squeeze(0), target_lang="eng_Latn")
            print(f"Original: {selected_sentence}")
            print(f"Predicted: {decoded_text}")
        except Exception as e:
            print(f"Decoding failed: {e}")
        
        print(f"Eval Loss: {eval_loss.item():.5f}")
        print(f"Eval Cosine Similarity: {eval_similarity.item():.4f}")
    
    return total_eval_loss, total_eval_similarity


eval_loss, eval_similarity = evaluate_model(lcm)

print("eval_similarity: ", eval_similarity)